# Chapter 8: Embeddings & Similarity Search

**Mathematics Behind LLMs — Book Series**

This chapter builds from first principles through the mathematics of embedding spaces, metric geometry,
contrastive learning losses, approximate nearest-neighbour search, dense retrieval training,
retrieval-augmented generation, and the Matryoshka Representation Learning objective.


## 8.0 Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")


## 8.1 Embedding Geometry

Given two embedding vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^d$, the three most common similarity / distance
measures are:

**L2 (Euclidean) distance**:
$$d_2(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2 = \sqrt{\sum_i (u_i - v_i)^2}$$

**Cosine similarity**:
$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \cdot \|\mathbf{v}\|_2}$$

**Inner product** (dot product): $\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i$

**Key identity**: for unit-norm vectors ($\|\mathbf{u}\|=\|\mathbf{v}\|=1$), cosine similarity equals inner product:

$$\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v} \quad \text{when } \|\mathbf{u}\| = \|\mathbf{v}\| = 1$$

Most modern retrieval systems (DPR, E5, GTE) L2-normalise their embeddings so that maximum inner product
search (MIPS) and maximum cosine similarity search become identical operations — enabling highly optimised
FAISS flat-IP indices.


In [ ]:
torch.manual_seed(42)
d = 128
u = torch.randn(d)
v = torch.randn(d)

# L2 distance
l2_dist = torch.dist(u, v).item()
# Equivalently via torch.cdist on batched inputs
l2_cdist = torch.cdist(u.unsqueeze(0), v.unsqueeze(0)).item()

# Cosine similarity
cos_sim = F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0)).item()

# Inner product
inner = (u * v).sum().item()

print(f"L2 distance    : {l2_dist:.4f}")
print(f"L2 via cdist   : {l2_cdist:.4f}")
print(f"Cosine sim     : {cos_sim:.4f}")
print(f"Inner product  : {inner:.4f}")

print()

# After L2 normalisation, cosine sim == inner product
u_norm = F.normalize(u, p=2, dim=0)
v_norm = F.normalize(v, p=2, dim=0)

cos_after = F.cosine_similarity(u_norm.unsqueeze(0), v_norm.unsqueeze(0)).item()
inner_after = (u_norm * v_norm).sum().item()

print("After F.normalize (L2):")
print(f"  ||u_norm||   : {u_norm.norm().item():.6f}")
print(f"  ||v_norm||   : {v_norm.norm().item():.6f}")
print(f"  cosine sim   : {cos_after:.6f}")
print(f"  inner product: {inner_after:.6f}")
print(f"  difference   : {abs(cos_after - inner_after):.2e}  <- should be ~0")

print()

# Batch pairwise distances and cosine similarities
B = 5  # batch
queries = torch.randn(B, d)
keys = torch.randn(B, d)
pairwise_l2 = torch.cdist(queries, keys)          # shape (B, B)
q_n = F.normalize(queries, p=2, dim=-1)
k_n = F.normalize(keys, p=2, dim=-1)
pairwise_cos = q_n @ k_n.T                        # shape (B, B)
print(f"Pairwise L2 matrix shape  : {pairwise_l2.shape}")
print(f"Pairwise cosine matrix shape: {pairwise_cos.shape}")
print(f"Cosine diagonal (query[i] vs key[i]):\n{pairwise_cos.diag()}")


## 8.2 InfoNCE Contrastive Loss

**InfoNCE (Noise Contrastive Estimation)** is the standard contrastive loss used in CLIP, SimCSE,
and dense retrieval training.  For a query $\mathbf{q}$, one positive key $\mathbf{k}^+$,
and $N-1$ negative keys $\{\mathbf{k}_i^-\}$:

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\mathbf{q} \cdot \mathbf{k}^+ / \tau)}{\exp(\mathbf{q} \cdot \mathbf{k}^+ / \tau) + \sum_{i=1}^{N-1} \exp(\mathbf{q} \cdot \mathbf{k}_i^- / \tau)}$$

where $\tau > 0$ is a **temperature** hyperparameter:
- Small $\tau$ (e.g., 0.05): sharper distribution, higher gradient signal for hard negatives
- Large $\tau$ (e.g., 1.0): softer distribution, more lenient training

This is equivalent to cross-entropy loss where the positive is the label and all keys form the class logits.
InfoNCE maximises a lower bound on mutual information $I(\mathbf{q}; \mathbf{k}^+)$.


In [ ]:
def infonce_loss(
    queries: torch.Tensor,      # (B, d)
    pos_keys: torch.Tensor,     # (B, d)
    neg_keys: torch.Tensor,     # (B, N_neg, d)
    temperature: float = 0.07,
) -> torch.Tensor:
    """
    InfoNCE loss from scratch.
    queries and pos_keys are L2-normalised inside the function.
    """
    B, d = queries.shape
    N_neg = neg_keys.shape[1]

    q = F.normalize(queries, p=2, dim=-1)            # (B, d)
    kp = F.normalize(pos_keys, p=2, dim=-1)          # (B, d)
    kn = F.normalize(neg_keys, p=2, dim=-1)          # (B, N_neg, d)

    # Positive similarities: (B,)
    pos_sim = (q * kp).sum(dim=-1) / temperature

    # Negative similarities: (B, N_neg)
    neg_sim = torch.bmm(kn, q.unsqueeze(-1)).squeeze(-1) / temperature  # (B, N_neg)

    # Numerically stable log-softmax style: stack pos as first column
    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)  # (B, 1+N_neg)
    labels = torch.zeros(B, dtype=torch.long)                   # positive is index 0
    loss = F.cross_entropy(logits, labels)
    return loss

torch.manual_seed(42)
B, d, N_neg = 32, 128, 63

# Random embeddings — loss should be close to log(1 + N_neg)
q_rand = torch.randn(B, d)
kp_rand = torch.randn(B, d)
kn_rand = torch.randn(B, N_neg, d)

loss_rand = infonce_loss(q_rand, kp_rand, kn_rand, temperature=0.07)
theoretical_max = math.log(1 + N_neg)
print(f"Random embeddings loss    : {loss_rand.item():.4f}")
print(f"Theoretical maximum (log(1+N_neg)): {theoretical_max:.4f}")

# Perfect retrieval: queries == positive keys
loss_perfect = infonce_loss(q_rand, q_rand.clone(), kn_rand, temperature=0.07)
print(f"Perfect alignment loss    : {loss_perfect.item():.4f}  (should be near 0)")

# Effect of temperature
print("\nTemperature sweep (random embeddings):")
for tau in [0.05, 0.1, 0.2, 0.5, 1.0]:
    l = infonce_loss(q_rand, kp_rand, kn_rand, temperature=tau).item()
    print(f"  tau={tau:.2f}  loss={l:.4f}")


## 8.3 Brute-Force k-Nearest Neighbours

The exact k-NN search scans the entire database for every query:

$$\text{kNN}(\mathbf{q}, \mathcal{D}, k) = \underset{\mathbf{x} \in \mathcal{D}}{\text{top-}k}\, \text{sim}(\mathbf{q}, \mathbf{x})$$

Time complexity per query: $O(Nd)$ where $N$ is the database size and $d$ is the embedding dimension.
For $N = 10^7$ and $d = 768$, this requires $\approx 7.7 \times 10^9$ multiply-accumulate operations per query —
at 10 ms per query, this allows only ~100 QPS on a single GPU.

Space complexity: $O(Nd)$ to store the database.

Brute-force is practical for $N \lesssim 10^5$.
For larger datasets, approximate nearest-neighbour (ANN) algorithms trade recall for speed.


In [ ]:
import time

def brute_force_knn(
    queries: torch.Tensor,    # (Q, d)
    database: torch.Tensor,   # (N, d)
    k: int,
) -> tuple:
    """Exact k-NN via pairwise L2 distance. Returns (distances, indices)."""
    # Use cosine similarity on normalised vectors (equivalent to inner product MIPS)
    q = F.normalize(queries, p=2, dim=-1)
    db = F.normalize(database, p=2, dim=-1)
    scores = q @ db.T                         # (Q, N)
    topk_scores, topk_idx = scores.topk(k, dim=-1, largest=True, sorted=True)
    return topk_scores, topk_idx

torch.manual_seed(0)
N, Q, d, k = 10_000, 100, 128, 10

database = torch.randn(N, d)
queries = torch.randn(Q, d)

t0 = time.perf_counter()
scores, indices = brute_force_knn(queries, database, k)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"N={N:,}  Q={Q}  d={d}  k={k}")
print(f"Brute-force k-NN time : {elapsed_ms:.2f} ms total  ({elapsed_ms/Q:.3f} ms/query)")
print(f"Top-k indices shape   : {indices.shape}")
print(f"Top-1 scores (first 5 queries): {scores[:5, 0].tolist()}")

# Extrapolate to larger N
for n_large in [1_000_000, 10_000_000]:
    # Scale linearly with N
    est_ms_per_query = (elapsed_ms / Q) * (n_large / N)
    est_qps = 1000 / est_ms_per_query
    print(f"Estimated @ N={n_large:,}: {est_ms_per_query:.1f} ms/query  ({est_qps:.0f} QPS)")


## 8.4 Random Projection LSH

**Locality Sensitive Hashing (LSH)** with random projections is a classic ANN method.
Each hash function is a random hyperplane:

$$h(\mathbf{x}) = \text{sign}(\mathbf{r} \cdot \mathbf{x}), \quad \mathbf{r} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

The probability that two vectors hash to the same bit depends only on the angle $\theta$ between them:

$$P(h(\mathbf{u}) = h(\mathbf{v})) = 1 - \frac{\theta(\mathbf{u}, \mathbf{v})}{\pi}$$

where $\theta = \arccos(\cos(\mathbf{u}, \mathbf{v}))$.

Using $b$ independent hash functions, the probability that **all** $b$ bits match is
$(1 - \theta/\pi)^b$, which falls off rapidly for dissimilar vectors.

In practice, multiple hash tables with different random projections are used to boost recall.


In [ ]:
class RandomProjectionLSH:
    """
    Simple random-projection LSH using b hyperplanes.
    Vectors that are similar (small angle) collide more often.
    """

    def __init__(self, d: int, b: int, seed: int = 0):
        """d = embedding dim, b = number of hash bits."""
        gen = torch.Generator()
        gen.manual_seed(seed)
        self.R = torch.randn(b, d, generator=gen)   # (b, d) projection matrix
        self.b = b

    def hash(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (N, d)  -> returns (N, b) binary hash codes (0/1 integers)
        """
        proj = x @ self.R.T          # (N, b)
        return (proj >= 0).int()     # sign -> {0, 1}

    def collision_rate(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        """Fraction of bits that match between two hash code matrices."""
        return (h1 == h2).float().mean(dim=-1)

torch.manual_seed(42)
d, b = 128, 32
lsh = RandomProjectionLSH(d, b)

# Create pairs with varying similarity
u = F.normalize(torch.randn(1, d), dim=-1)
similarities = [1.0, 0.9, 0.7, 0.5, 0.0, -0.5]
print(f"{'Cosine sim':>12}  {'Expected P(collision)':>22}  {'Empirical P(collision)':>22}")
print("-" * 60)
for sim in similarities:
    # Construct v with target cosine similarity to u
    perp = F.normalize(torch.randn(1, d), dim=-1)
    perp = perp - (perp * u).sum(dim=-1, keepdim=True) * u  # orthogonalise
    perp = F.normalize(perp, dim=-1)
    cos_sim_val = max(-1.0, min(1.0, sim))
    v = cos_sim_val * u + math.sqrt(max(0.0, 1.0 - cos_sim_val**2)) * perp
    v = F.normalize(v, dim=-1)

    actual_cos = F.cosine_similarity(u, v).item()
    theta = math.acos(max(-1.0, min(1.0, actual_cos)))
    expected_p = 1.0 - theta / math.pi

    h_u = lsh.hash(u)
    h_v = lsh.hash(v)
    empirical_p = lsh.collision_rate(h_u, h_v).item()

    print(f"{actual_cos:>12.3f}  {expected_p:>22.4f}  {empirical_p:>22.4f}")


## 8.5 Product Quantization (PQ)

**Product Quantization** dramatically reduces memory for large vector databases.
A $d$-dimensional vector is split into $m$ equal sub-vectors of size $d/m$.
Each sub-vector is independently quantized to one of $k$ centroids learned by k-means.

**Storage**:  each vector is stored as $m$ integers in $[0, k)$, requiring $m \lceil \log_2 k \rceil$ bits
vs $32d$ bits for FP32.  With $m=8$, $k=256$ (1 byte per sub-code), $d=128$:

$$\text{PQ bits} = 8 \times 8 = 64 \text{ bits}, \quad \text{FP32 bits} = 32 \times 128 = 4096 \text{ bits}$$

Compression ratio: $4096 / 64 = 64\times$.

**Asymmetric distance computation (ADC)**: precompute sub-vector distances between the query and all
$k$ centroids per sub-space ($m \times k$ table), then look up and sum distances in $O(m)$ per candidate.


In [ ]:
class ProductQuantizer:
    """
    Simple Product Quantizer: trains m sub-space k-means codebooks.
    Uses random centroid init for speed (not full k-means++ in demo).
    """

    def __init__(self, d: int, m: int, k: int, n_iter: int = 20, seed: int = 0):
        assert d % m == 0, "d must be divisible by m"
        self.d = d
        self.m = m
        self.k = k
        self.sub_d = d // m
        self.n_iter = n_iter
        self.seed = seed
        self.codebooks = None  # list of m tensors each (k, sub_d)

    def _kmeans(self, X: torch.Tensor) -> torch.Tensor:
        """Mini k-means: X is (N, sub_d), returns centroids (k, sub_d)."""
        torch.manual_seed(self.seed)
        idx = torch.randperm(X.shape[0])[:self.k]
        centroids = X[idx].clone()
        for _ in range(self.n_iter):
            dists = torch.cdist(X, centroids)              # (N, k)
            assignments = dists.argmin(dim=-1)             # (N,)
            new_centroids = torch.zeros_like(centroids)
            counts = torch.zeros(self.k)
            new_centroids.scatter_add_(0, assignments.unsqueeze(1).expand_as(X), X)
            counts.scatter_add_(0, assignments, torch.ones(X.shape[0]))
            mask = counts > 0
            new_centroids[mask] = new_centroids[mask] / counts[mask].unsqueeze(1)
            new_centroids[~mask] = centroids[~mask]  # keep old centroid if empty
            centroids = new_centroids
        return centroids

    def fit(self, X: torch.Tensor):
        """X: (N, d)"""
        self.codebooks = []
        for i in range(self.m):
            sub = X[:, i * self.sub_d:(i + 1) * self.sub_d]
            cb = self._kmeans(sub)
            self.codebooks.append(cb)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        """X: (N, d) -> codes (N, m) as uint8-range integers."""
        codes = []
        for i, cb in enumerate(self.codebooks):
            sub = X[:, i * self.sub_d:(i + 1) * self.sub_d]
            dists = torch.cdist(sub, cb)         # (N, k)
            codes.append(dists.argmin(dim=-1))   # (N,)
        return torch.stack(codes, dim=1)          # (N, m)

    def decode(self, codes: torch.Tensor) -> torch.Tensor:
        """codes: (N, m) -> reconstructed (N, d)."""
        parts = []
        for i, cb in enumerate(self.codebooks):
            parts.append(cb[codes[:, i]])  # (N, sub_d)
        return torch.cat(parts, dim=1)

torch.manual_seed(0)
N_train, N_test, d, m, k = 2000, 100, 32, 4, 16

X_train = torch.randn(N_train, d)
X_test  = torch.randn(N_test, d)

pq = ProductQuantizer(d=d, m=m, k=k)
pq.fit(X_train)

# Encode and decode
codes = pq.encode(X_test)          # (N_test, m)
X_recon = pq.decode(codes)         # (N_test, d)

recon_err = (X_test - X_recon).norm(dim=-1).mean().item()
print(f"PQ config: d={d}, m={m}, k={k}  (sub_d={d//m})")
print(f"Mean reconstruction L2 error: {recon_err:.4f}")

# Storage comparison
fp32_bits = 32 * d
pq_bits = m * math.ceil(math.log2(k))
print(f"FP32 bits per vector  : {fp32_bits}")
print(f"PQ bits per vector    : {pq_bits}")
print(f"Compression ratio     : {fp32_bits / pq_bits:.1f}x")

# Recall@1 on training set
X_enc = pq.encode(X_train)
X_approx = pq.decode(X_enc)
# For each X_train vector, check if its nearest neighbour in X_approx is itself
exact_dists = torch.cdist(X_train, X_train)
approx_dists = torch.cdist(X_train, X_approx)
exact_nn = exact_dists.topk(2, largest=False).indices[:, 1]   # exclude self
approx_nn = approx_dists.topk(2, largest=False).indices[:, 1]
recall_at_1 = (exact_nn == approx_nn).float().mean().item()
print(f"Recall@1 (PQ approx vs exact): {recall_at_1:.4f}")


## 8.6 Bi-Encoder Training (DPR)

**Dense Passage Retrieval (DPR)** trains two separate encoders — one for queries, one for passages —
using a contrastive loss with in-batch negatives.

For a batch of $B$ (query, positive-passage) pairs, the loss for query $i$ is:

$$\mathcal{L}_i = -\log \frac{e^{\mathbf{q}_i^\top \mathbf{p}_i^+}}{e^{\mathbf{q}_i^\top \mathbf{p}_i^+} + \sum_{j \neq i} e^{\mathbf{q}_i^\top \mathbf{p}_j^+} + \sum_m e^{\mathbf{q}_i^\top \mathbf{p}_m^-}}$$

The positive for query $i$ is passage $i$ (diagonal of the batch score matrix).
All other in-batch positives serve as **hard negatives** — they are relevant to their own queries
but not to query $i$, making them challenging negatives without requiring explicit negative mining.

The total batch loss is $\mathcal{L} = \frac{1}{B}\sum_i \mathcal{L}_i$, equivalent to cross-entropy
with targets $= \text{diag}(\{0, 1, \ldots, B-1\})$.


In [ ]:
def dpr_loss(
    query_embs: torch.Tensor,    # (B, d) — already from query encoder
    pos_embs: torch.Tensor,      # (B, d) — positive passage embeddings
    neg_embs: torch.Tensor,      # (B, K, d) — explicit hard negatives (optional, may be empty)
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    DPR contrastive loss with in-batch negatives.
    Diagonal of (query x pos) score matrix = positives.
    Off-diagonal + explicit negatives = negatives.
    """
    B, d = query_embs.shape
    q = F.normalize(query_embs, p=2, dim=-1)      # (B, d)
    p = F.normalize(pos_embs, p=2, dim=-1)         # (B, d)

    # In-batch scores: (B, B)
    scores = (q @ p.T) / temperature

    if neg_embs is not None and neg_embs.numel() > 0:
        K = neg_embs.shape[1]
        n = F.normalize(neg_embs, p=2, dim=-1)     # (B, K, d)
        # Score each query against its own K explicit negatives
        neg_scores = torch.bmm(n, q.unsqueeze(-1)).squeeze(-1) / temperature  # (B, K)
        # Append negatives to score matrix
        scores = torch.cat([scores, neg_scores], dim=1)  # (B, B+K)

    # Label: positive for query i is index i (diagonal)
    labels = torch.arange(B)
    return F.cross_entropy(scores, labels)

torch.manual_seed(0)
B, d, K = 16, 128, 4

q_embs = torch.randn(B, d)
p_embs_pos = torch.randn(B, d)
p_embs_neg = torch.randn(B, K, d)

loss_rand = dpr_loss(q_embs, p_embs_pos, p_embs_neg)
print(f"DPR loss (random)   : {loss_rand.item():.4f}")
print(f"Expected (log(B+K)) : {math.log(B + K):.4f}")

# Perfect retrieval: query == positive
loss_perfect = dpr_loss(q_embs, q_embs.clone(), p_embs_neg)
print(f"DPR loss (perfect)  : {loss_perfect.item():.4f}  (should be near 0)")

# Verify that in-batch negatives create a harder task than explicit negatives alone
# (using more negatives drives the loss up toward log(total_negatives))
for b_size in [4, 8, 16, 32]:
    q_ = torch.randn(b_size, d)
    p_ = torch.randn(b_size, d)
    l_ = dpr_loss(q_, p_, None)
    print(f"  B={b_size:2d}  loss={l_.item():.4f}  log(B)={math.log(b_size):.4f}")


## 8.7 RAG Marginalization

**Retrieval-Augmented Generation (RAG)** marginalizes over the top-$k$ retrieved documents
to produce the final output probability:

$$P_{\text{RAG}}(y \mid x) = \sum_{z \in \text{top-}k} p_\eta(z \mid x) \cdot p_\theta(y \mid x, z)$$

where:
- $p_\eta(z \mid x)$: retriever score for document $z$ given query $x$, normalised over top-$k$
- $p_\theta(y \mid x, z)$: generator log-probability of output $y$ given input $x$ and document $z$

For numerical stability, work in log-space using the **log-sum-exp** identity:

$$\log P_{\text{RAG}}(y \mid x) = \log \sum_{z} \exp\bigl(\log p_\eta(z \mid x) + \log p_\theta(y \mid x, z)\bigr)$$

This is computed with `torch.logsumexp` to avoid overflow.


In [ ]:
def rag_marginalize(
    retriever_scores: torch.Tensor,   # (B, k) raw dot-product retriever scores
    generator_logprobs: torch.Tensor, # (B, k) log P(y | x, z_i) from LM
) -> torch.Tensor:
    """
    Compute log P_RAG(y | x) = log sum_z [ p_eta(z|x) * p_theta(y|x,z) ]
    using the log-sum-exp trick for numerical stability.

    Returns: (B,) log-probabilities of the output y for each input x.
    """
    # Normalise retriever scores to log-probabilities over top-k docs
    log_retriever = F.log_softmax(retriever_scores, dim=-1)   # (B, k)

    # Sum log-probs in log-space: log(p_eta * p_theta) = log_p_eta + log_p_theta
    log_joint = log_retriever + generator_logprobs             # (B, k)

    # Marginalise over documents: log sum_z exp(log_joint)
    log_rag = torch.logsumexp(log_joint, dim=-1)               # (B,)
    return log_rag

torch.manual_seed(0)
B, k = 4, 5  # 4 queries, 5 retrieved documents each

# Simulated retriever scores and LM log-probs
retriever_scores = torch.randn(B, k)        # e.g., dot-product similarity
generator_logprobs = -torch.rand(B, k) * 3 # log P in (-3, 0)

log_rag = rag_marginalize(retriever_scores, generator_logprobs)

print("Retriever scores (raw):")
print(retriever_scores)
print("\nGenerator log-probs:")
print(generator_logprobs)
print("\nRAG log P(y|x) per query:")
print(log_rag)
print("\nRAG P(y|x) per query:")
print(log_rag.exp())

# Verify: naive computation matches logsumexp result
log_ret_norm = F.log_softmax(retriever_scores, dim=-1)
naive_sum = (log_ret_norm.exp() * generator_logprobs.exp()).sum(dim=-1).log()
print(f"\nMax diff (logsumexp vs naive): {(log_rag - naive_sum).abs().max().item():.2e}")

# Show logsumexp stability with extreme values
extreme_scores = torch.tensor([[1000.0, 999.0, 998.0]])
extreme_logprobs = torch.tensor([[-0.1, -0.2, -0.5]])
log_rag_extreme = rag_marginalize(extreme_scores, extreme_logprobs)
print(f"\nExtreme scores RAG log-prob: {log_rag_extreme.item():.4f}  (no overflow/NaN)")


## 8.8 Matryoshka Representation Learning (MRL)

**Matryoshka Representation Learning** trains an embedding model so that every
**prefix** of the embedding vector is also a useful, independently capable representation.

Given nesting dimensions $M = \{d_1, d_2, \ldots, d_L\}$ with $d_1 < d_2 < \ldots < d_L = d$,
and a loss function $\mathcal{L}$ (e.g., InfoNCE), the MRL objective is:

$$\mathcal{L}_{\text{MRL}} = \sum_{d' \in M} \frac{1}{|M|} \cdot \mathcal{L}\bigl(f_{d'}(\mathbf{x})\bigr)$$

where $f_{d'}(\mathbf{x}) = \mathbf{z}_{1:d'}$ is the first $d'$ dimensions of the full embedding $\mathbf{z}$.

This allows a single model to serve multiple deployment scenarios:
- Use $d' = 64$ for high-throughput, memory-constrained retrieval
- Use $d' = 512$ for maximum accuracy
- No retraining or separate models needed

Typical nesting set: $M = \{8, 16, 32, 64, 128, 256, 512\}$.


In [ ]:
class MRLLoss(nn.Module):
    """
    Matryoshka Representation Learning loss.
    Applies InfoNCE at each nested dimension d' in `nesting_dims`.
    """

    def __init__(self, nesting_dims: list, temperature: float = 0.07):
        super().__init__()
        self.nesting_dims = sorted(nesting_dims)
        self.temperature = temperature

    def _infonce(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        """In-batch InfoNCE: q (B,d'), k (B,d')."""
        q = F.normalize(q, p=2, dim=-1)
        k = F.normalize(k, p=2, dim=-1)
        scores = (q @ k.T) / self.temperature   # (B, B)
        labels = torch.arange(q.shape[0])
        return F.cross_entropy(scores, labels)

    def forward(
        self,
        query_embs: torch.Tensor,   # (B, d_full)
        key_embs: torch.Tensor,     # (B, d_full)
    ) -> torch.Tensor:
        total_loss = torch.tensor(0.0, requires_grad=True)
        for d_prime in self.nesting_dims:
            loss_dp = self._infonce(query_embs[:, :d_prime], key_embs[:, :d_prime])
            total_loss = total_loss + loss_dp / len(self.nesting_dims)
        return total_loss


class SmallEncoder(nn.Module):
    """Toy encoder: input_dim -> d_full via two linear layers."""

    def __init__(self, input_dim: int, d_full: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, d_full * 2),
            nn.ReLU(),
            nn.Linear(d_full * 2, d_full),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)  # (B, d_full)


torch.manual_seed(0)
input_dim = 64
d_full = 128
nesting_dims = [8, 16, 32, 64, 128]
B = 32

encoder = SmallEncoder(input_dim, d_full)
mrl_loss_fn = MRLLoss(nesting_dims=nesting_dims, temperature=0.07)
optimizer = torch.optim.Adam(encoder.parameters(), lr=3e-4)

# Simulate a tiny training loop
# Positive pairs: (x, augmented_x) with Gaussian noise as augmentation
print("Training with MRL loss:")
for step in range(200):
    x = torch.randn(B, input_dim)
    x_aug = x + 0.1 * torch.randn_like(x)   # simple noise augmentation

    q_emb = encoder(x)
    k_emb = encoder(x_aug)

    loss = mrl_loss_fn(q_emb, k_emb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0 or step == 199:
        print(f"  step {step:3d}  loss = {loss.item():.4f}")

print()

# Verify that prefix-64 still retrieves correctly
encoder.eval()
with torch.no_grad():
    x_eval = torch.randn(100, input_dim)
    x_aug_eval = x_eval + 0.1 * torch.randn_like(x_eval)
    q_full = F.normalize(encoder(x_eval), dim=-1)
    k_full = F.normalize(encoder(x_aug_eval), dim=-1)

    for d_prime in nesting_dims:
        q_ = F.normalize(q_full[:, :d_prime], dim=-1)
        k_ = F.normalize(k_full[:, :d_prime], dim=-1)
        scores = q_ @ k_.T  # (100, 100)
        preds = scores.argmax(dim=-1)
        targets = torch.arange(100)
        recall_at_1 = (preds == targets).float().mean().item()
        print(f"  prefix-{d_prime:3d}  Recall@1 = {recall_at_1:.4f}")


## 8.9 Summary

| Topic                       | Key Takeaway                                                              |
|-----------------------------|--------------------------------------------------------------------------|
| Embedding geometry          | After L2-norm, cosine sim = inner product; use F.normalize before index |
| InfoNCE loss                | Cross-entropy over similarity logits; temperature controls sharpness     |
| Brute-force k-NN            | Exact but $O(Nd)$; impractical for $N > 10^5$                           |
| Random projection LSH       | Collision probability = $1 - \theta/\pi$; near-exact for cosine sim     |
| Product quantization        | Split + k-means sub-codes; 32–64x compression with small recall cost     |
| DPR bi-encoder              | In-batch negatives; cross-entropy on diagonal = positive signal          |
| RAG marginalization         | Log-sum-exp over top-k retrieved docs; numerically stable                |
| Matryoshka (MRL)            | Average InfoNCE at each prefix dimension; single model for all sizes     |

Embedding systems sit at the intersection of metric geometry, information theory, and approximate
algorithms.  Understanding the mathematical properties of each component — from the cosine-inner-product
identity to the Kahan compensation trick in gradient accumulation — is essential for building
scalable, accurate retrieval pipelines for production LLM deployments.
